# NB01: ENIGMA Environmental-Microbe Census

**Goal**: Build the complete linkage table mapping ENIGMA field sites to geochemistry measurements and microbial community profiles. Determine the scale and coverage of environmental-microbe linkage data.

**Requires**: BERDL JupyterHub (Spark access)

**Key questions**:
1. How many samples have BOTH geochemistry AND community profiles?
2. What chemical species are measured and at what coverage?
3. What taxonomic resolution is available for community members?
4. How many isolated genomes link back to environmental samples?

**Outputs**: `data/enigma_linkage_summary.csv`, `data/enigma_geochem_coverage.csv`, `data/enigma_community_geochem_overlap.csv`, coverage figures

In [ ]:
# Initialize Spark session
spark = get_spark_session()
spark.sql("SELECT 1 AS test").show()
print("Spark session ready")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data")
FIG_DIR = Path("../figures")
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 150, "figure.figsize": (12, 6)})

## 1. ENIGMA Field Sites and Samples

In [ ]:
# Load all locations
locations = spark.sql("""
    SELECT sdt_location_name, latitude_degree, longitude_degree,
           continent_sys_oterm_name, country_sys_oterm_name,
           region, biome_sys_oterm_name, feature_sys_oterm_name
    FROM enigma_coral.sdt_location
""").toPandas()

print(f"Total locations: {len(locations)}")
print(f"\nBiomes: {locations['biome_sys_oterm_name'].nunique()}")
print(locations['biome_sys_oterm_name'].value_counts().head(10))
print(f"\nFeatures: {locations['feature_sys_oterm_name'].nunique()}")
print(locations['feature_sys_oterm_name'].value_counts().head(10))

In [ ]:
# Load all samples with location joins
samples = spark.sql("""
    SELECT s.sdt_sample_name, s.sdt_location_name, s.depth_meter,
           s.elevation_meter, s.date, s.time,
           s.material_sys_oterm_name, s.env_package_sys_oterm_name,
           l.latitude_degree, l.longitude_degree,
           l.biome_sys_oterm_name, l.feature_sys_oterm_name, l.region
    FROM enigma_coral.sdt_sample s
    LEFT JOIN enigma_coral.sdt_location l
        ON s.sdt_location_name = l.sdt_location_name
""").toPandas()

print(f"Total samples: {len(samples)}")
print(f"Samples with location link: {samples['latitude_degree'].notna().sum()}")
print(f"Unique locations sampled: {samples['sdt_location_name'].nunique()}")
print(f"Date range: {samples['date'].min()} to {samples['date'].max()}")
print(f"\nMaterial types:")
print(samples['material_sys_oterm_name'].value_counts().head(10))

## 2. Geochemistry Coverage

What chemical species are measured, how many samples have each, and what concentration ranges?

In [ ]:
# Load geochemistry data -- aggregate replicates per sample-molecule
geochem = spark.sql("""
    SELECT sdt_sample_name,
           molecule_from_list_sys_oterm_name AS molecule,
           physiochemical_state,
           AVG(concentration_micromolar) AS mean_conc_uM,
           COUNT(*) AS n_replicates
    FROM enigma_coral.ddt_brick0000010
    GROUP BY sdt_sample_name, molecule_from_list_sys_oterm_name, physiochemical_state
""").toPandas()

print(f"Total geochem records (after replicate averaging): {len(geochem)}")
print(f"Unique samples with geochem: {geochem['sdt_sample_name'].nunique()}")
print(f"Chemical species measured: {geochem['molecule'].nunique()}")
print(f"Physiochemical states: {geochem['physiochemical_state'].unique().tolist()}")

In [ ]:
# Per-molecule coverage: how many samples have each chemical species measured?
geochem_coverage = (
    geochem
    .groupby("molecule")
    .agg(
        n_samples=("sdt_sample_name", "nunique"),
        mean_conc=("mean_conc_uM", "mean"),
        median_conc=("mean_conc_uM", "median"),
        max_conc=("mean_conc_uM", "max"),
        min_conc=("mean_conc_uM", "min"),
    )
    .sort_values("n_samples", ascending=False)
    .reset_index()
)

# Classify molecules
ree_elements = [
    "cerium", "dysprosium atom", "erbium", "europium atom", "gadolinium atom",
    "holmium atom", "lanthanum atom", "lutetium atom", "neodymium atom",
    "praseodymium atom", "samarium atom", "thulium atom", "ytterbium"
]
nitrogen_compounds = ["nitrate", "nitrite"]
critical_metals = [
    "cobalt atom", "nickel atom", "manganese atom", "chromium atom",
    "tungsten", "molybdenum atom", "vanadium atom", "gallium atom",
    "germanium atom"
]

geochem_coverage["category"] = "Other metal"
geochem_coverage.loc[geochem_coverage["molecule"].isin(ree_elements), "category"] = "Rare Earth Element"
geochem_coverage.loc[geochem_coverage["molecule"].isin(nitrogen_compounds), "category"] = "Nitrogen compound"
geochem_coverage.loc[geochem_coverage["molecule"].isin(critical_metals), "category"] = "Critical mineral"

print("=== Geochemistry Coverage Summary ===\n")
for cat in ["Rare Earth Element", "Critical mineral", "Nitrogen compound", "Other metal"]:
    subset = geochem_coverage[geochem_coverage["category"] == cat]
    print(f"\n{cat}s ({len(subset)}):")
    for _, row in subset.iterrows():
        print(f"  {row['molecule']:30s}  samples={row['n_samples']:4d}  "
              f"range=[{row['min_conc']:.3g}, {row['max_conc']:.3g}] µM")

geochem_coverage.to_csv(DATA_DIR / "enigma_geochem_coverage.csv", index=False)
print(f"\nSaved to {DATA_DIR / 'enigma_geochem_coverage.csv'}")

In [ ]:
# Figure: Geochemistry coverage heatmap by molecule category
fig, ax = plt.subplots(figsize=(14, 8))
plot_data = geochem_coverage.sort_values(["category", "n_samples"], ascending=[True, False])
colors = {"Rare Earth Element": "#e74c3c", "Critical mineral": "#3498db",
          "Nitrogen compound": "#2ecc71", "Other metal": "#95a5a6"}
bar_colors = [colors[c] for c in plot_data["category"]]

ax.barh(range(len(plot_data)), plot_data["n_samples"], color=bar_colors)
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data["molecule"], fontsize=8)
ax.set_xlabel("Number of samples with measurement")
ax.set_title("ENIGMA Geochemistry: Sample Coverage by Chemical Species")
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in colors.items()]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.savefig(FIG_DIR / "enigma_geochem_coverage.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {FIG_DIR / 'enigma_geochem_coverage.png'}")

## 3. Microbial Community Profiles

Load community-sample linkages and ASV abundance/taxonomy data.

In [ ]:
# Load community-sample linkages
communities = spark.sql("""
    SELECT sdt_community_name, sdt_sample_name,
           community_type_sys_oterm_name, sdt_condition_name,
           sdt_community_description
    FROM enigma_coral.sdt_community
""").toPandas()

print(f"Total communities: {len(communities)}")
print(f"Communities with sample link: {communities['sdt_sample_name'].notna().sum()}")
print(f"Unique samples with communities: {communities['sdt_sample_name'].nunique()}")
print(f"\nCommunity types:")
print(communities['community_type_sys_oterm_name'].value_counts())

In [ ]:
# ASV abundance: how many ASVs per community?
asv_stats = spark.sql("""
    SELECT sdt_community_name,
           COUNT(DISTINCT sdt_asv_name) AS n_asvs,
           SUM(count_count_unit) AS total_reads
    FROM enigma_coral.ddt_brick0000459
    WHERE count_count_unit > 0
    GROUP BY sdt_community_name
""").toPandas()

print(f"Communities with ASV data: {len(asv_stats)}")
print(f"Total ASV-community associations: {asv_stats['n_asvs'].sum()}")
print(f"\nASVs per community:")
print(asv_stats['n_asvs'].describe())

In [ ]:
# Taxonomy: what levels are available?
tax_levels = spark.sql("""
    SELECT taxonomic_level_sys_oterm_name AS level,
           COUNT(DISTINCT sdt_asv_name) AS n_asvs,
           COUNT(DISTINCT sdt_taxon_name) AS n_taxa
    FROM enigma_coral.ddt_brick0000454
    GROUP BY taxonomic_level_sys_oterm_name
    ORDER BY n_asvs DESC
""").toPandas()

print("=== Taxonomy Resolution ===")
print(tax_levels.to_string(index=False))

# Get genus-level diversity
genus_diversity = spark.sql("""
    SELECT sdt_taxon_name AS genus, COUNT(DISTINCT sdt_asv_name) AS n_asvs
    FROM enigma_coral.ddt_brick0000454
    WHERE taxonomic_level_sys_oterm_name = 'Genus'
    GROUP BY sdt_taxon_name
    ORDER BY n_asvs DESC
""").toPandas()

print(f"\nUnique genera: {len(genus_diversity)}")
print(f"\nTop 20 genera by ASV count:")
print(genus_diversity.head(20).to_string(index=False))

## 4. The Critical Overlap: Samples with BOTH Geochemistry AND Community Profiles

This is the key linkage -- samples where we know both what chemicals are present and what microbes are there.

In [ ]:
# Find the overlap: samples with both geochemistry and community profiles
geochem_samples = set(geochem["sdt_sample_name"].unique())
community_samples = set(communities["sdt_sample_name"].dropna().unique())
all_samples = set(samples["sdt_sample_name"].unique())

overlap = geochem_samples & community_samples
geochem_only = geochem_samples - community_samples
community_only = community_samples - geochem_samples

print("=== Sample Overlap Analysis ===")
print(f"Total unique samples:           {len(all_samples)}")
print(f"Samples with geochemistry:      {len(geochem_samples)}")
print(f"Samples with community profile: {len(community_samples)}")
print(f"Samples with BOTH:              {len(overlap)}  <-- key linkage")
print(f"Geochemistry only:              {len(geochem_only)}")
print(f"Community only:                 {len(community_only)}")
print(f"\nOverlap rate: {len(overlap)/len(geochem_samples)*100:.1f}% of geochem samples also have community data")

In [ ]:
# Figure: Venn diagram of sample data types
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Venn-style bar chart
categories = ["Geochem\nonly", "BOTH\n(key linkage)", "Community\nonly"]
counts = [len(geochem_only), len(overlap), len(community_only)]
bar_colors = ["#3498db", "#e74c3c", "#2ecc71"]
axes[0].bar(categories, counts, color=bar_colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Number of samples")
axes[0].set_title("ENIGMA: Sample Data Type Overlap")
for i, (c, v) in enumerate(zip(categories, counts)):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Right: For overlap samples, how many molecules are measured per sample?
overlap_geochem = geochem[geochem["sdt_sample_name"].isin(overlap)]
mols_per_sample = overlap_geochem.groupby("sdt_sample_name")["molecule"].nunique()
axes[1].hist(mols_per_sample, bins=30, color="#e74c3c", edgecolor="black", alpha=0.8)
axes[1].set_xlabel("Number of chemical species measured")
axes[1].set_ylabel("Number of overlap samples")
axes[1].set_title(f"Chemical Richness in {len(overlap)} Overlap Samples")
axes[1].axvline(mols_per_sample.median(), color="black", linestyle="--",
                label=f"Median = {mols_per_sample.median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "enigma_sample_overlap.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Isolated Strains and Genomes

How many ENIGMA isolates have sequenced genomes, and can they be linked back to environmental samples?

In [ ]:
# Load strain and genome data
strains = spark.sql("""
    SELECT sdt_strain_name, sdt_strain_description, sdt_genome_name
    FROM enigma_coral.sdt_strain
""").toPandas()

genomes = spark.sql("""
    SELECT sdt_genome_name, sdt_strain_name, n_contigs_count_unit, n_features_count_unit
    FROM enigma_coral.sdt_genome
""").toPandas()

print(f"Total strains: {len(strains)}")
print(f"Strains with genome link: {strains['sdt_genome_name'].notna().sum()}")
print(f"Total genomes: {len(genomes)}")
print(f"Genomes with strain link: {genomes['sdt_strain_name'].notna().sum()}")

# Check genome quality
genomes_quality = spark.sql("""
    SELECT sdt_strain_name,
           genome_completeness_method_checkm_percent AS completeness,
           genome_contamination_method_checkm_percent AS contamination,
           genome_n50_method_checkm_count_unit AS n50
    FROM enigma_coral.ddt_brick0000521
""").toPandas()

print(f"\nGenomes with quality metrics: {len(genomes_quality)}")
if len(genomes_quality) > 0:
    hq = genomes_quality[
        (genomes_quality["completeness"] >= 90) &
        (genomes_quality["contamination"] <= 5)
    ]
    print(f"High-quality genomes (>90% complete, <5% contamination): {len(hq)}")

## 6. Additional ENIGMA Data Tables

Check what other data tables are available (TnSeq, DubSeq, condition metadata, etc.)

In [ ]:
# Survey all ENIGMA tables: row counts and column counts
enigma_tables = spark.sql("SHOW TABLES IN enigma_coral").toPandas()
print(f"Total ENIGMA tables: {len(enigma_tables)}")

table_stats = []
for _, row in enigma_tables.iterrows():
    tname = row["tableName"]
    try:
        count = spark.sql(f"SELECT COUNT(*) AS n FROM enigma_coral.{tname}").collect()[0]["n"]
        cols = spark.sql(f"DESCRIBE enigma_coral.{tname}").count()
        table_stats.append({"table": tname, "rows": count, "columns": cols})
    except Exception as e:
        table_stats.append({"table": tname, "rows": -1, "columns": -1})

table_stats_df = pd.DataFrame(table_stats).sort_values("rows", ascending=False)
print("\n=== ENIGMA CORAL Table Inventory ===")
print(table_stats_df.to_string(index=False))

In [ ]:
# Check conditions table -- may have additional experimental metadata
conditions = spark.sql("""
    SELECT sdt_condition_name, sdt_condition_id
    FROM enigma_coral.sdt_condition
    LIMIT 20
""").toPandas()
print(f"Conditions sample:\n{conditions}")

# Check TnSeq libraries -- may have fitness data for ENIGMA isolates
tnseq = spark.sql("""
    SELECT COUNT(*) AS n FROM enigma_coral.sdt_tnseq_library
""").collect()[0]["n"]
dubseq = spark.sql("""
    SELECT COUNT(*) AS n FROM enigma_coral.sdt_dubseq_library
""").collect()[0]["n"]
print(f"\nTnSeq libraries: {tnseq}")
print(f"DubSeq libraries: {dubseq}")

## 7. Build Complete Linkage Table

Join everything: Location -> Sample -> Geochemistry + Community -> ASV -> Taxonomy

In [ ]:
# Build the master linkage table using Spark for the heavy joins
# For overlap samples: join geochemistry (wide format) with community genus-level composition

# Step 1: Pivot geochemistry to wide format (one row per sample, one col per molecule)
# Use only Supernatant measurements (dissolved fraction, most relevant for bioavailability)
geochem_wide = spark.sql("""
    SELECT sdt_sample_name,
           molecule_from_list_sys_oterm_name AS molecule,
           AVG(concentration_micromolar) AS conc_uM
    FROM enigma_coral.ddt_brick0000010
    WHERE physiochemical_state = 'Supernatant'
    GROUP BY sdt_sample_name, molecule_from_list_sys_oterm_name
""")

# Pivot to wide format
geochem_pivot = geochem_wide.groupBy("sdt_sample_name").pivot("molecule").agg({"conc_uM": "first"})
geochem_pivot_pd = geochem_pivot.toPandas()
print(f"Geochemistry wide table: {geochem_pivot_pd.shape[0]} samples x {geochem_pivot_pd.shape[1]} columns")
print(f"Non-null coverage per molecule:")
coverage = geochem_pivot_pd.drop(columns=["sdt_sample_name"]).notna().sum().sort_values(ascending=False)
print(coverage.head(20))

In [ ]:
# Step 2: Build genus-level community composition for overlap samples
# Join community -> ASV abundance -> ASV taxonomy (genus level)
overlap_list = list(overlap)
if len(overlap_list) > 0:
    # Create temp view with overlap sample names
    overlap_df = spark.createDataFrame([(s,) for s in overlap_list], ["sdt_sample_name"])
    overlap_df.createOrReplaceTempView("overlap_samples")

    community_genus = spark.sql("""
        SELECT c.sdt_sample_name,
               t.sdt_taxon_name AS genus,
               SUM(a.count_count_unit) AS total_reads
        FROM enigma_coral.sdt_community c
        JOIN enigma_coral.ddt_brick0000459 a
            ON c.sdt_community_name = a.sdt_community_name
        JOIN enigma_coral.ddt_brick0000454 t
            ON a.sdt_asv_name = t.sdt_asv_name
        JOIN overlap_samples o
            ON c.sdt_sample_name = o.sdt_sample_name
        WHERE t.taxonomic_level_sys_oterm_name = 'Genus'
          AND a.count_count_unit > 0
        GROUP BY c.sdt_sample_name, t.sdt_taxon_name
    """).toPandas()

    print(f"Community genus composition: {len(community_genus)} sample-genus records")
    print(f"Unique samples: {community_genus['sdt_sample_name'].nunique()}")
    print(f"Unique genera: {community_genus['genus'].nunique()}")

    # Compute relative abundance per sample
    sample_totals = community_genus.groupby("sdt_sample_name")["total_reads"].sum()
    community_genus["rel_abundance"] = community_genus.apply(
        lambda r: r["total_reads"] / sample_totals[r["sdt_sample_name"]], axis=1
    )

    # Save
    community_genus.to_csv(DATA_DIR / "enigma_community_genus_overlap.csv", index=False)
    print(f"\nSaved to {DATA_DIR / 'enigma_community_genus_overlap.csv'}")
else:
    print("WARNING: No overlap samples found -- check sample name matching")

## 8. Summary Statistics and Data Export

In [ ]:
# Build comprehensive linkage summary
linkage_summary = {
    "ENIGMA Locations": len(locations),
    "ENIGMA Samples (total)": len(samples),
    "Samples with geochemistry": len(geochem_samples),
    "Samples with community profiles": len(community_samples),
    "Samples with BOTH (key linkage)": len(overlap),
    "Chemical species measured": geochem["molecule"].nunique(),
    "  - Rare earth elements": len([m for m in geochem["molecule"].unique() if m in ree_elements]),
    "  - Critical minerals": len([m for m in geochem["molecule"].unique() if m in critical_metals]),
    "  - Nitrogen compounds": len([m for m in geochem["molecule"].unique() if m in nitrogen_compounds]),
    "Unique ASVs": asv_stats["n_asvs"].sum() if len(asv_stats) > 0 else 0,
    "Communities with ASV data": len(asv_stats),
    "Isolated strains": len(strains),
    "Sequenced genomes": len(genomes),
    "Genomes with quality metrics": len(genomes_quality),
}

summary_df = pd.DataFrame(
    [{"metric": k, "value": v} for k, v in linkage_summary.items()]
)
summary_df.to_csv(DATA_DIR / "enigma_linkage_summary.csv", index=False)

print("=== ENIGMA Environmental-Microbe Linkage Summary ===\n")
for k, v in linkage_summary.items():
    print(f"  {k:45s} {v:>8,}")

# Save geochem pivot for downstream use
geochem_pivot_pd.to_csv(DATA_DIR / "enigma_geochem_wide.csv", index=False)
print(f"\nAll data files saved to {DATA_DIR}/")

## 9. Key Findings & Next Steps

**Summary**: This notebook inventoried all ENIGMA CORAL environmental-microbe linkage data. Key metrics:
- Number of samples with both geochemistry AND community data (the "overlap" set)
- Chemical species coverage across those samples
- Taxonomic resolution (genus-level from 16S)
- Available isolated genomes

**Next**: NB02 will perform the same census for NMDC multi-omics data, and NB03 will characterize pangenome environmental metadata coverage.